# Stage 3 — AWGN Channel + Power Normalization

Stage 2 proved the autoencoder compresses+reconstructs with the channel off. Now insert
a real wireless channel: **additive white Gaussian noise (AWGN)**, `y = x + w`, plus the
**power normalization** every transmitter needs — a radio can't send unlimited energy,
so the network's channel symbols must obey a fixed average-power budget before they hit
the noise.

**Concepts:**
- **Power constraint** `E|x_k|^2 = 1` per complex symbol — a fair, fixed energy budget,
  independent of what the encoder learned. `deepscs/channel.py::power_norm` rescales
  each frame's symbols to hit this exactly (see docs/initial_plan.md #5).
- **SNR (signal-to-noise ratio)**, in dB: how much stronger the signal is than the
  noise floor. Higher SNR = clearer channel. `snr_db_to_sigma` converts an SNR target
  into the per-real-dimension noise standard deviation, given unit signal power.
- **Differentiable noise**: `w` is sampled with `torch.randn`, which is a valid
  operation inside autograd — gradients flow encoder <- channel <- decoder, so the
  network can learn to be *robust* to noise, not just to compress.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from IPython.display import Audio, display

from deepscs.audio import SR, W, F, L, deframe, sdr
from deepscs.data import ToyDataset
from deepscs.model import DeepSC_S
from deepscs.channel import ChannelLayer, to_iq, power_norm, measured_power, snr_db_to_sigma
from deepscs.viz import plot_waveform, plot_constellation

torch.manual_seed(0)
np.random.seed(0)
plt.rcParams["figure.figsize"] = (10, 3)

# CPU, not MPS: MPS's autograd for torch.complex ops (used by the channel layer)
# doesn't support backward through non-contiguous views yet; toy-scale training
# (<=32 clips, 115k params) is fast enough on CPU that it doesn't matter here.
device = torch.device("cpu")
print("device:", device)

## Power normalization — verify the constraint

Run a batch through `SemanticEncoder -> ChannelEncoder`, then `power_norm`. The measured average power per complex symbol should land within 1% of 1.0 — this is success criterion #2's power half.

In [ ]:
model = DeepSC_S(depth=8, n_blocks=4).to(device)
train_ds = ToyDataset(n=4, length=W, seed=0)
waveforms = torch.stack([train_ds[i] for i in range(len(train_ds))]).to(device)
x = waveforms.view(-1, 1, F, L)

with torch.no_grad():
    z = model.channel_encoder(model.semantic_encoder(x))
    x_iq_raw = to_iq(z)
    x_iq = power_norm(x_iq_raw)

p_before = measured_power(x_iq_raw)
p_after = measured_power(x_iq)
print(f"measured power before norm: {p_before:.4f}")
print(f"measured power after norm:  {p_after:.4f}  (target 1.0)")
assert abs(p_after - 1.0) < 0.01, "power constraint violated"
print("power constraint holds within 1%.")

## Constellation — transmitted symbols

Each complex channel symbol is one point `(I, Q)`. Before any noise, they should spread across a disk of roughly unit average power (not necessarily unit *magnitude* per symbol — that's an average over all symbols, not per symbol).

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
plot_constellation(ax, x_iq.cpu().numpy(), title="transmitted symbols X (no noise yet)")
plt.show()

## SNR sweep — noise cloud growth

At low SNR the additive noise is comparable in magnitude to the signal itself, so `Y = X + W` scatters far from the original constellation points; at high SNR the cloud tightens back down around `X`.

In [ ]:
channel = ChannelLayer("awgn", snr_db=0.0)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, snr in zip(axes, [0, 6, 12, 18]):
    channel.snr_db = snr
    with torch.no_grad():
        y = channel.impl(x_iq, snr)
    plot_constellation(ax, y.cpu().numpy(), title=f"Y at {snr} dB SNR")
plt.tight_layout()
plt.show()

for snr in [0, 6, 12, 18]:
    sigma = snr_db_to_sigma(snr)
    print(f"SNR={snr:>3} dB -> per-real-dim noise std sigma={sigma:.4f}")

## Train across an SNR range

Train the full transceiver (channel now *enabled*) with SNR sampled uniformly at
random each step from 0-20 dB (docs #CHOICE — random-uniform training SNR gives one
model that's robust across the whole range, instead of overfitting to a single
operating point).

In [ ]:
model = DeepSC_S(depth=8, n_blocks=4).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()
channel = ChannelLayer("awgn", snr_db=10.0)

losses = []
n_epochs = 1500
for epoch in range(n_epochs):
    channel.snr_db = float(np.random.uniform(0, 20))
    opt.zero_grad()
    x_hat = model(x, channel=channel)
    loss = loss_fn(x_hat, x)
    loss.backward()
    opt.step()
    losses.append(loss.item())

print(f"loss[0]={losses[0]:.5f}  loss[-1]={losses[-1]:.5f}")

fig, ax = plt.subplots()
ax.plot(losses)
ax.set_yscale("log")
ax.set_xlabel("step")
ax.set_ylabel("MSE loss")
ax.set_title("training loss, SNR ~ U(0,20) dB")
plt.show()

## SDR vs SNR

Evaluate the trained model at fixed SNRs. Reconstruction quality should degrade smoothly as SNR drops and improve as it rises — success criterion #2.

In [ ]:
model.eval()
snr_range = list(range(-5, 21, 2.5))
mean_sdrs = []
for snr in snr_range:
    channel.snr_db = snr
    with torch.no_grad():
        x_hat = model(x, channel=channel)
    clip_sdrs = [
        sdr(deframe(x[i, 0].cpu().numpy()), deframe(x_hat[i, 0].cpu().numpy()))
        for i in range(x.shape[0])
    ]
    mean_sdrs.append(np.mean(clip_sdrs))
    print(f"SNR={snr:>5.1f} dB -> mean SDR={mean_sdrs[-1]:.2f} dB")

fig, ax = plt.subplots()
ax.plot(snr_range, mean_sdrs, marker="o")
ax.set_xlabel("channel SNR (dB)")
ax.set_ylabel("reconstruction SDR (dB)")
ax.set_title("SDR vs SNR (AWGN)")
ax.grid(alpha=0.3)
plt.show()

deltas = np.diff(mean_sdrs)
print("SDR monotonically non-decreasing with SNR:", bool(np.all(deltas > -1.5)))

**Experiment — fixed 0 dB, listen.** Retrain briefly at a single harsh SNR and listen to what survives: at 0 dB the noise power equals the signal power, so fine detail (consonants, sibilants — high-frequency, low-energy transients) tends to wash out first, while the low-frequency voiced/vowel energy survives longest, since it carries more of the signal's total power and dominates what the MSE loss protects.

In [ ]:
fixed_model = DeepSC_S(depth=8, n_blocks=4).to(device)
fixed_opt = torch.optim.Adam(fixed_model.parameters(), lr=1e-3)
fixed_channel = ChannelLayer("awgn", snr_db=0.0)

for epoch in range(800):
    fixed_opt.zero_grad()
    x_hat = fixed_model(x, channel=fixed_channel)
    loss = loss_fn(x_hat, x)
    loss.backward()
    fixed_opt.step()

fixed_model.eval()
with torch.no_grad():
    x_hat = fixed_model(x, channel=fixed_channel)

i = 0
orig = deframe(x[i, 0].cpu().numpy())
recon = deframe(x_hat[i, 0].cpu().numpy())
print(f"SDR at fixed 0 dB training/eval: {sdr(orig, recon):.2f} dB")

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
plot_waveform(axes[0], orig, title="original")
plot_waveform(axes[1], recon, title="reconstruction (trained+evaluated at 0 dB)")
plt.tight_layout()
plt.show()

print("original:")
display(Audio(orig, rate=SR))
print("reconstruction at 0 dB:")
display(Audio(recon, rate=SR))

## Verify (success criterion #2)

In [ ]:
assert abs(measured_power(power_norm(to_iq(model.channel_encoder(model.semantic_encoder(x))))) - 1.0) < 0.01
assert np.all(np.diff(mean_sdrs) > -1.5), "SDR should not collapse as SNR rises"
print("power constraint holds; SDR degrades gracefully with SNR. Stage 3 verified.")